# Extended Lab: Regularization for Wine Quality Classification

**Dataset:** `wine_quality.csv` (1,599 red wines, 11 physicochemical features, binary `quality` label: 0 = lower quality, 1 = higher quality)

**Builds on:** the Codecademy "Regularization in Machine Learning" lesson (bias-variance tradeoff, L1/Lasso, L2/Ridge, hyperparameter tuning, `GridSearchCV`) and the original `sol.py` walkthrough — extended with EDA, a full model comparison, ROC/AUC analysis, and an Elastic Net extension that the original script did not cover.

## Learning objectives
By the end of this notebook you will be able to:
1. Diagnose overfitting by comparing training vs. test performance.
2. Explain *why* regularization requires feature scaling.
3. Implement unregularized, L2 (Ridge/default), L1 (Lasso), and Elastic Net logistic regression in scikit-learn.
4. Tune the regularization hyperparameter `C` with `GridSearchCV` and `LogisticRegressionCV`.
5. Interpret coefficient plots to see L1's feature-selection behavior vs. L2's shrinkage behavior.
6. Compare models on multiple metrics (F1, accuracy, precision, recall, ROC-AUC) and justify a final choice.

> 💡 Reminder: in `LogisticRegression`, the hyperparameter is **`C`, the inverse of regularization strength** (`C = 1/α`). Small `C` → strong regularization; large `C` → weak regularization. This is the opposite convention from `Ridge`/`Lasso` for linear regression, which use `alpha` directly.

## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (9, 5)

df = pd.read_csv('wine_quality.csv')
print(df.shape)
df.head()

### Checkpoint 0.1 — First look
Check for missing values and the balance of the target class. An imbalanced target changes how we should interpret accuracy vs. F1.

In [ ]:
print(df.isna().sum())
print()
print(df['quality'].value_counts(normalize=True))
sns.countplot(x='quality', data=df)
plt.title('Class balance: quality (0 = lower, 1 = higher)')
plt.show()

**Discussion:** The classes are reasonably balanced (≈53%/47%), so accuracy is a defensible metric here, but F1 (which balances precision and recall) is still the safer default when reporting results — it is what the original lesson script uses.

## Part 1 — Exploratory Data Analysis

In [ ]:
corr = df.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()

**Checkpoint 1.1:** Identify any pairs of features with |correlation| > 0.6 (excluding the diagonal). These are candidates for multicollinearity — one of the classic symptoms of overfitting that regularization helps address.

In [ ]:
high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .rename('correlation')
        .reset_index()
)
high_corr_pairs = high_corr_pairs[high_corr_pairs['correlation'].abs() > 0.6]
high_corr_pairs.sort_values('correlation', key=abs, ascending=False)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.ravel(), df.columns):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Part 2 — Preprocessing: why scaling is *required* for regularization

Both the L1 penalty (`α·Σ|b_j|`) and the L2 penalty (`α·Σb_j²`) operate on the *raw magnitude* of coefficients. If `total sulfur dioxide` (values in the hundreds) and `chlorides` (values near 0.08) are on wildly different scales, their coefficients will be on wildly different scales too — and the penalty will punish them unequally for reasons that have nothing to do with predictive importance. `StandardScaler` puts every feature on a mean-0, variance-1 footing so the penalty is fair across features.

In [ ]:
from sklearn.preprocessing import StandardScaler

y = df['quality']
features = df.drop(columns=['quality'])
predictors = features.columns

scaler = StandardScaler().fit(features)
X = scaler.transform(features)

## Part 3 — Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=99, stratify=y
)
print(X_train.shape, X_test.shape)

> Note: we added `stratify=y` (not in the original script) so both splits preserve the ~53/47 class balance — good practice whenever you can afford it.

## Part 4 — Baseline: unregularized logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

clf_no_reg = LogisticRegression(penalty=None, max_iter=5000)
clf_no_reg.fit(X_train, y_train)

y_pred_train = clf_no_reg.predict(X_train)
y_pred_test = clf_no_reg.predict(X_test)

print('Training F1:', f1_score(y_train, y_pred_train))
print('Testing  F1:', f1_score(y_test, y_pred_test))

In [ ]:
coef = pd.Series(clf_no_reg.coef_.ravel(), predictors).sort_values()
coef.plot(kind='bar', title='Coefficients (no regularization)')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

**Checkpoint 4.1:** Look at the gap between training and test F1, and note any coefficients with surprisingly large magnitude relative to the others. Do you see signs of overfitting? Given the correlation pairs from Part 1, which coefficients might be unstable?

## Part 5 — Default scikit-learn logistic regression (L2-regularized!)

`LogisticRegression()` defaults to `penalty='l2'` with `C=1.0`. This is already a regularized model — a detail that surprises many people coming from linear regression's `LinearRegression()`, which has no such default.

In [ ]:
clf_default = LogisticRegression(max_iter=5000)
clf_default.fit(X_train, y_train)

y_pred_train_ridge = clf_default.predict(X_train)
y_pred_test_ridge = clf_default.predict(X_test)

print('Ridge-regularized training F1:', f1_score(y_train, y_pred_train_ridge))
print('Ridge-regularized testing  F1:', f1_score(y_test, y_pred_test_ridge))

## Part 6 — Coarse-grained hyperparameter search over C

In [ ]:
C_array = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
training_array, test_array = [], []

for c in C_array:
    clf = LogisticRegression(C=c, max_iter=5000)
    clf.fit(X_train, y_train)
    training_array.append(f1_score(y_train, clf.predict(X_train)))
    test_array.append(f1_score(y_test, clf.predict(X_test)))

plt.plot(C_array, training_array, marker='o', label='Training F1')
plt.plot(C_array, test_array, marker='o', label='Test F1')
plt.xscale('log')
plt.xlabel('C (inverse regularization strength)')
plt.ylabel('F1 score')
plt.legend()
plt.title('F1 vs. C')
plt.show()

**Checkpoint 6.1:** Where does the test curve peak? Note the rough region of C — this tells you what range to hand to `GridSearchCV` next.

## Part 7 — Fine-grained search with GridSearchCV (L2)

In [ ]:
from sklearn.model_selection import GridSearchCV

C_array_fine = np.logspace(-4, 2, 100)
tuning_C = {'C': C_array_fine}

clf_gs = LogisticRegression(max_iter=5000)
gs = GridSearchCV(clf_gs, param_grid=tuning_C, scoring='f1', cv=5)
gs.fit(X_train, y_train)

print('Best C:', gs.best_params_['C'])
print('Best CV F1:', gs.best_score_)

In [ ]:
clf_best = LogisticRegression(C=gs.best_params_['C'], max_iter=5000)
clf_best.fit(X_train, y_train)
y_pred_best = clf_best.predict(X_test)
print('Held-out test F1 with tuned C:', f1_score(y_test, y_pred_best))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best)
plt.title('Tuned L2 model — confusion matrix')
plt.show()

## Part 8 — L1 (Lasso) regularization with LogisticRegressionCV

L1 requires the `liblinear` or `saga` solver. `LogisticRegressionCV` performs the C search internally and is faster than wiring up `GridSearchCV` by hand.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

C_array_l1 = np.logspace(-2, 2, 100)
clf_l1 = LogisticRegressionCV(
    Cs=C_array_l1, cv=5, penalty='l1', solver='liblinear',
    scoring='f1', max_iter=5000
)
clf_l1.fit(X, y)   # fit on the full scaled dataset, as in the original script

print('Best C:', clf_l1.C_)
print('Best fit coefficients:', clf_l1.coef_)

In [ ]:
coef_l1 = pd.Series(clf_l1.coef_.ravel(), predictors).sort_values()
coef_l1.plot(kind='bar', title='Coefficients for tuned L1 (Lasso)')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

zeroed = coef_l1[coef_l1 == 0]
print(f"L1 eliminated {len(zeroed)} of {len(coef_l1)} features:", list(zeroed.index))

**Checkpoint 8.1:** Compare this bar chart to the no-regularization plot in Part 4. Which features survived L1's feature selection? Does that match the highly-correlated pairs you found in Part 1 (L1 tends to keep one feature from a correlated pair and zero out the other)?

## Part 9 — Extension: Elastic Net

Elastic Net blends L1 and L2 penalties via `l1_ratio` (1.0 = pure L1, 0.0 = pure L2). It needs the `saga` solver and is useful when you want *some* sparsity without L1's sometimes-arbitrary choice among correlated features.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

clf_enet = LogisticRegressionCV(
    Cs=np.logspace(-2, 2, 20),
    l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
    penalty='elasticnet', solver='saga', cv=5,
    scoring='f1', max_iter=5000, random_state=99
)
clf_enet.fit(X_train, y_train)

print('Best C:', clf_enet.C_)
print('Best l1_ratio:', clf_enet.l1_ratio_)
y_pred_enet = clf_enet.predict(X_test)
print('Test F1:', f1_score(y_test, y_pred_enet))

## Part 10 — Model comparison table

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score

models = {
    'No regularization': clf_no_reg,
    'L2 default (C=1)': clf_default,
    'L2 tuned (GridSearchCV)': clf_best,
    'Elastic Net tuned': clf_enet,
}

rows = []
for name, m in models.items():
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, proba),
    })

# L1 model was fit on the full X, so evaluate its test-fold predictions separately for fairness
pred_l1 = clf_l1.predict(X_test)
proba_l1 = clf_l1.predict_proba(X_test)[:, 1]
rows.append({
    'Model': 'L1 tuned (LogisticRegressionCV)',
    'Accuracy': accuracy_score(y_test, pred_l1),
    'Precision': precision_score(y_test, pred_l1),
    'Recall': recall_score(y_test, pred_l1),
    'F1': f1_score(y_test, pred_l1),
    'ROC-AUC': roc_auc_score(y_test, proba_l1),
})

comparison = pd.DataFrame(rows).set_index('Model').round(3)
comparison

> ⚠️ Caveat: the L1 model above was fit on `(X, y)` (full dataset) to mirror the original lesson script, so its test-set score is optimistic — the test rows were seen during its training. In your own work, always fit every model on `X_train, y_train` only and hold `X_test` out consistently, as done for the other models.

## Part 11 — ROC curves

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 7))
for name, m in models.items():
    RocCurveDisplay.from_estimator(m, X_test, y_test, ax=ax, name=name)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('ROC curves — regularization comparison')
plt.show()

## Part 12 — Reflection

Answer these in a markdown cell of your own:

1. Which model would you deploy, and why — F1, ROC-AUC, and interpretability all matter differently depending on the business context.
2. If you had 200 features instead of 11, would you lean L1 or L2? Why?
3. Two features had correlation > 0.6 in Part 1. Trace what happened to their coefficients across the no-regularization, L2, and L1 plots. What does this tell you about how each penalty handles multicollinearity?
4. `alpha` and `C` are inverses (`C = 1/alpha`). Sanity check: does increasing `C` in this notebook increase or decrease the training F1 score? Does that match theory?
5. What would you expect to change if you used `roc_auc` instead of `f1` as the `GridSearchCV` scoring metric?